# Whisper-large-v3 LoRA Fine-Tuning on Children's Speech (Kaggle)

This notebook fine-tunes `openai/whisper-large-v3` with LoRA on the Pasketti competition data.
Uses INT8 quantization + LoRA (r=32, alpha=64) to fit on Kaggle T4 GPU.

**Requirements:**
- Kaggle dataset with competition audio uploaded as a private dataset
- HuggingFace Hub token (set as Kaggle secret `HF_TOKEN`)
- ~8 hours GPU quota

## 1. Setup

In [ ]:
import subprocess
import sys

# Install dependencies not pre-installed on Kaggle
deps = ["jiwer>=3.0", "audiomentations>=0.36", "peft>=0.12", "bitsandbytes>=0.43"]
for dep in deps:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(".").resolve()
if (PROJECT_ROOT / "src").exists():
    sys.path.insert(0, str(PROJECT_ROOT))
elif Path("/kaggle/input/childwhisper-src/src").exists():
    sys.path.insert(0, "/kaggle/input/childwhisper-src")
    PROJECT_ROOT = Path("/kaggle/input/childwhisper-src")

from src.kaggle_utils import (
    get_paths_lora,
    setup_hub_auth,
    get_latest_checkpoint,
    download_checkpoint,
    get_lora_training_args,
    verify_kaggle_data,
    check_gpu_memory,
    is_kaggle,
)
from src.train_whisper_lora import main as train_lora_main

print(f"Running on Kaggle: {is_kaggle()}")
print(f"Project root: {PROJECT_ROOT}")

## 2. Environment & Paths

In [ ]:
# Configure paths based on environment
DATASET_SLUG = "pasketti-word-audio"
LOCAL_DATA_DIR = "data"

paths = get_paths_lora(dataset_slug=DATASET_SLUG, local_data_dir=LOCAL_DATA_DIR)
AUDIO_DIR = paths["audio_dir"]
METADATA_PATH = paths["metadata_path"]
OUTPUT_DIR = paths["output_dir"]

CONFIG_PATH = PROJECT_ROOT / "configs" / "training_config.yaml"
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path("configs/training_config.yaml")

print(f"Audio dir:     {AUDIO_DIR}")
print(f"Metadata:      {METADATA_PATH}")
print(f"Output dir:    {OUTPUT_DIR}")
print(f"Config:        {CONFIG_PATH}")

## 3. Data Verification

In [ ]:
stats = verify_kaggle_data(str(AUDIO_DIR), str(METADATA_PATH))

print(f"Utterances:       {stats['num_utterances']}")
print(f"Audio found:      {stats['num_audio_found']}")
print(f"Audio missing:    {stats['num_missing_audio']}")

if stats["duration_stats"]:
    ds = stats["duration_stats"]
    print("\nDuration stats:")
    print(f"  Min:          {ds['min']:.2f}s")
    print(f"  Max:          {ds['max']:.2f}s")
    print(f"  Mean:         {ds['mean']:.2f}s")
    print(f"  Total hours:  {ds['total_hours']:.1f}h")

if stats["num_missing_audio"] > 0:
    pct = stats["num_missing_audio"] / stats["num_utterances"] * 100
    print(f"\nWARNING: {pct:.1f}% of audio files are missing!")

## 4. HuggingFace Hub Authentication

In [ ]:
HUB_MODEL_ID = "nishantgaurav23/pasketti-whisper-lora"
PUSH_TO_HUB = True

try:
    setup_hub_auth()
    print("HF Hub authenticated successfully")
except ValueError as e:
    print(f"Hub auth skipped: {e}")
    print("Checkpoints will be saved locally only")
    PUSH_TO_HUB = False

## 5. Training Config & GPU Check

In [ ]:
import yaml

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

wl = config.get("whisper_large_v3", {})
common = config.get("common", {})
lora = wl.get("lora", {})

print("=== Whisper-large-v3 LoRA Training Config ===")
print(f"Model:            {wl.get('model_name', 'openai/whisper-large-v3')}")
print(f"Learning rate:    {wl.get('learning_rate', 1e-3)}")
print(f"Epochs:           {wl.get('num_train_epochs', 3)}")
print(f"Batch size:       {wl.get('per_device_train_batch_size', 1)}")
print(f"Grad accum:       {wl.get('gradient_accumulation_steps', 16)}")
print(f"Effective batch:  {wl.get('per_device_train_batch_size', 1) * wl.get('gradient_accumulation_steps', 16)}")
print(f"FP16:             {wl.get('fp16', True)}")
print(f"INT8 quant:       {wl.get('load_in_8bit', True)}")
print(f"LoRA r:           {lora.get('r', 32)}")
print(f"LoRA alpha:       {lora.get('alpha', 64)}")
print(f"LoRA targets:     {lora.get('target_modules', ['q_proj', 'v_proj'])}")
print(f"SpecAugment:      {common.get('spec_augment', {}).get('apply', True)}")
print(f"Hub model ID:     {wl.get('hub_model_id', 'N/A')}")

print("\n=== GPU Info ===")
gpu_info = check_gpu_memory()
print(f"GPU:              {gpu_info['gpu_name'] or 'None (CPU only)'}")
print(f"Memory:           {gpu_info['total_memory_gb']:.1f} GB")
print(f"Sufficient:       {gpu_info['is_sufficient']}")
if not gpu_info["is_sufficient"]:
    print("WARNING: GPU memory may be insufficient for INT8+LoRA training!")

## 6. Resume Check

In [ ]:
RESUME_FROM = None

if PUSH_TO_HUB and get_latest_checkpoint(HUB_MODEL_ID):
    print(f"Found existing adapter at {HUB_MODEL_ID}")
    resume_dir = str(OUTPUT_DIR / "resume")
    downloaded = download_checkpoint(HUB_MODEL_ID, resume_dir)
    if downloaded:
        RESUME_FROM = downloaded
        print(f"Downloaded adapter to {RESUME_FROM}")
    else:
        print("Failed to download, starting fresh")
else:
    print("No existing adapter found, training from scratch")

## 7. Train

In [ ]:
import time

train_args = get_lora_training_args(
    config_path=str(CONFIG_PATH),
    metadata_path=str(METADATA_PATH),
    audio_dir=str(AUDIO_DIR),
    output_dir=str(OUTPUT_DIR),
    resume_from=RESUME_FROM,
)

print(f"Training args: {' '.join(train_args)}")
print("\nStarting LoRA training...")

start_time = time.time()
val_wer = train_lora_main(train_args)
elapsed = time.time() - start_time

print("\nTraining complete!")
print(f"Validation WER: {val_wer:.4f}")
print(f"Elapsed time: {elapsed / 60:.1f} minutes")

## 8. Evaluate

In [ ]:
print("=== LoRA Training Summary ===")
print(f"Final Validation WER: {val_wer:.4f}")
print(f"Training time:        {elapsed / 60:.1f} min ({elapsed / 3600:.1f} hrs)")
print(f"Output directory:     {OUTPUT_DIR}")

if val_wer < 0.17:
    print("\nExcellent! Target WER (<0.17) achieved with large-v3 LoRA!")
elif val_wer < 0.20:
    print("\nGood progress. Consider noise augmentation for further improvement.")
else:
    print("\nWER still high. Check data quality, LoRA rank, or learning rate.")

## 9. Save LoRA Adapter to HuggingFace Hub

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi

    api = HfApi()
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        create_pr=False,
    )
    print(f"LoRA adapter uploaded to https://huggingface.co/{HUB_MODEL_ID}")
else:
    print(f"Hub push disabled. Adapter saved locally at {OUTPUT_DIR}")
    print("To upload manually: huggingface-cli upload-folder ...")